In [7]:
import csv
from pathlib import Path

In [8]:
scheduled_path = Path("raw-zip-scheduled")
scheduled_files = sorted(scheduled_path.glob("*.zip"))
print(len(scheduled_files))
scheduled_files[:5]

218


[PosixPath('raw-zip-scheduled/20080101SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080301SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080401SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080501SCLineOutages_csv.zip')]

In [9]:
import re
from collections import defaultdict, namedtuple

ScheduledOutage = namedtuple(
    "ScheduledOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "scheduled_out_datetime",
        "scheduled_in_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)
Interval = namedtuple("Interval", ["start", "end"])

equipment_name_pattern = r"^([A-Za-z0-9._ -]{8})-([A-Za-z0-9._ -]{8})_(\d{2,3})_(.+)$"

In [10]:
zip_path = scheduled_files[0]

In [11]:
import io
from datetime import datetime
from zipfile import ZipFile


def parse_scheduled_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ScheduledOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        scheduled_out_datetime=datetime.strptime(
            row["Scheduled Out Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        scheduled_in_datetime=datetime.strptime(
            row["Scheduled In Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def list_csvs(zip_path):
    with ZipFile(zip_path) as z:
        return [i for i in sorted(z.namelist()) if i.endswith(".csv")]


def read_csv_from_zip(zip_path, csv_name):
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            data = f.read()
        csv_reader = csv.DictReader(io.StringIO(data.decode("utf-8")))
        # Only keep rows with line outages
        data = [parse_scheduled_outage(row) for row in csv_reader]
        data = [row for row in data if row is not None]
    return data


def summarize_data(data):
    """Implement processing method described in this paper titled "Transmission grid outage statistics extracted from a webpage logging outages in northeast america" """
    # group by ptid
    by_ptid = defaultdict(list)
    for outage in data:
        by_ptid[outage.ptid].append(outage)
    # save (in, out) dates for each ptid
    summary = defaultdict(set)

    for ptid, rows in by_ptid.items():
        for row in rows:
            key = (
                row.scheduled_in_datetime,
                row.scheduled_out_datetime,
                row.equipment_name,
                row.bus1,
                row.bus2,
                row.voltage,
            )
            summary[ptid].add(key)
    return summary


# Example using an existing variable in the notebook:
zip_path = scheduled_files[0]
print(zip_path)
print("members:", list_csvs(zip_path))

# Read one file from the zip (text)
member = list_csvs(zip_path)[0]
data = read_csv_from_zip(zip_path, member)
print(len(data))
summary = summarize_data(data)
print(len(summary))

raw-zip-scheduled/20080101SCLineOutages_csv.zip
members: ['20080101SCLineOutages.csv', '20080102SCLineOutages.csv', '20080103SCLineOutages.csv', '20080104SCLineOutages.csv', '20080105SCLineOutages.csv', '20080106SCLineOutages.csv', '20080107SCLineOutages.csv', '20080108SCLineOutages.csv', '20080109SCLineOutages.csv', '20080110SCLineOutages.csv', '20080111SCLineOutages.csv', '20080112SCLineOutages.csv', '20080113SCLineOutages.csv', '20080114SCLineOutages.csv', '20080115SCLineOutages.csv', '20080116SCLineOutages.csv', '20080117SCLineOutages.csv', '20080118SCLineOutages.csv', '20080119SCLineOutages.csv', '20080120SCLineOutages.csv', '20080121SCLineOutages.csv', '20080122SCLineOutages.csv', '20080123SCLineOutages.csv', '20080124SCLineOutages.csv', '20080125SCLineOutages.csv', '20080126SCLineOutages.csv', '20080127SCLineOutages.csv', '20080128SCLineOutages.csv', '20080129SCLineOutages.csv', '20080130SCLineOutages.csv', '20080131SCLineOutages.csv']
6376
24


In [12]:
from tqdm import tqdm

scheduled_outages = defaultdict(set)
for zip_path in tqdm(scheduled_files):
    for member in list_csvs(zip_path):
        data = read_csv_from_zip(zip_path, member)
        summary = summarize_data(data)
        for ptid, in_out_set in summary.items():
            scheduled_outages[ptid].update(in_out_set)

100%|██████████| 218/218 [20:25<00:00,  5.62s/it]


In [13]:
scheduled_outages.keys()

dict_keys([25035, 25042, 25050, 25094, 25167, 25169, 25243, 25561, 25562, 25563, 25564, 25876, 26020, 26030, 26053, 26135, 26187, 26261, 26477, 26478, 26507, 26582, 25304, 25313, 25565, 25566, 26480, 25574, 325223, 25296, 26425, 26092, 26158, 25544, 25158, 26159, 25053, 25060, 26055, 25139, 25141, 25137, 25017, 25289, 25323, 25547, 25319, 25771, 25269, 26102, 25320, 25198, 25857, 325582, 26236, 26427, 25321, 26027, 26031, 25015, 25426, 26022, 26023, 26201, 26109, 25731, 25732, 26048, 25511, 25542, 25312, 25518, 25519, 25520, 25521, 25884, 26084, 25190, 25228, 25725, 26461, 25177, 25290, 25324, 25308, 26452, 25301, 25275, 25264, 25349, 25212, 25537, 25554, 25533, 25061, 325166, 25012, 25538, 25556, 25871, 26182, 25536, 26021, 25222, 25251, 25303, 25220, 68810, 26229, 26240, 26247, 25285, 25769, 25770, 25253, 25510, 25581, 25582, 25583, 25270, 25494, 25020, 25145, 25341, 325240, 25568, 25550, 26209, 25181, 26624, 26213, 25150, 25155, 25126, 25300, 25299, 26123, 25244, 26691, 25283, 32523

In [14]:
type(scheduled_outages[25013])

set

In [15]:
scheduled_outages[25013]

{(datetime.datetime(2016, 12, 14, 6, 59),
  datetime.datetime(2016, 11, 25, 16, 22),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2016, 12, 2, 17, 59),
  datetime.datetime(2016, 11, 25, 16, 22),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2016, 7, 5, 15, 59),
  datetime.datetime(2016, 7, 5, 9, 51),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2016, 8, 3, 19, 59),
  datetime.datetime(2016, 8, 3, 12, 47),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2018, 10, 20, 20, 59),
  datetime.datetime(2018, 10, 12, 8, 6),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2013, 12, 15, 22, 59, 59),
  datetime.datetime(2013, 12, 14, 2, 12, 6),
  'E.SAYRE_-NWAVERLY_115_956',
  'E.SAYRE_',
  'NWAVERLY',
  115),
 (datetime.datetime(2016, 7, 5, 21, 59),
  datetime.datetime(2016, 7, 5, 9, 51),
  'E.SAY

In [18]:
import pandas as pd

actual_outage_csv_path = Path("processed-scheduled-outages.csv")

# build a flat table from the nested defaultdict
rows = []
for ptid, schedule_set in scheduled_outages.items():
    for info_tuple in sorted(schedule_set, key=lambda x: x[0]):
        rows.append(
            {
                "PTID": ptid,
                "Name": info_tuple[2],
                "Scheduled In Date/Time": info_tuple[0],
                "Scheduled Out Date/Time": info_tuple[1],
                # "Voltage": info_tuple[5],
                # "FirstBus": info_tuple[3],
                # "SecondBus": info_tuple[4],
            }
        )

rows = sorted(rows, key=lambda x: x["Scheduled In Date/Time"])
df = pd.DataFrame(rows)

# write out; use the existing json path with a .csv suffix
df.to_csv(actual_outage_csv_path, index=False)

print(f"saved {len(df)} rows to {actual_outage_csv_path}")

saved 10354716 rows to processed-scheduled-outages.csv
